# Deep Reinforcement Learning - Lab Assignment 1
## Part #1 - Multi-Armed Bandit (MAB)
### Adaptive Treatment Recommendation System using Multi-Armed Bandit Learning

**Team / Group Number (G): 178**

Each medicine is modelled as an *arm* of a Multi-Armed Bandit. The system learns from
patient outcomes over time and progressively identifies the optimal medicine.

Strategies implemented:
1. **Greedy / Immediate Exploitation** (Task 2)
2. **Epsilon-Greedy / Controlled Clinical Trial** (Task 3)
3. **UCB1 / Confidence-Based** (Task 4)
4. **Comparative Analysis** (Task 5)

### Execution metadata (Virtual Lab requirement)
The cell below prints the **execution timestamp** and **Virtual Machine / Host ID**.
A timestamped screenshot from the virtual lab must accompany the final submission.

In [ ]:
# Print execution timestamp and Virtual Machine ID at the top of the notebook
import datetime          # standard library for the current date/time
import socket            # used to read the machine/host name (acts as VM ID)
import platform          # extra environment details for traceability

# Current wall-clock time of execution (match this with the virtual-lab screenshot)
print('Execution Timestamp :', datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
# Hostname acts as the Virtual Machine identifier inside the virtual lab
print('Virtual Machine ID  :', socket.gethostname())
# Platform info for full reproducibility context
print('Platform            :', platform.platform())
print('Python Version      :', platform.python_version())

## Task 1: Dataset Design (1 Mark)

The synthetic patient-treatment environment is derived from the group number `G = 178`
so the dataset is unique and reproducible.

**Derivation for G = 178**
- Medicines: `K = (G mod 3) + 5 = 1 + 5 = 6`
- Hidden success probability: `P_i = 0.4 + ((G + i) mod 6) * 0.07`
- Severity: `Severity = (patient_id mod 5) + 1`  (1 = mild ... 5 = critical)
- Utility (reward): `utility = clinical_outcome * (1 - Severity/10)`

In [ ]:
# ---- Core imports ----
import random                       # python RNG (seeded for reproducibility)
import numpy as np                  # vectorised maths and RNG
import pandas as pd                 # tabular dataset handling
import matplotlib.pyplot as plt     # plotting cumulative-reward curves

# ---- Reproducibility: seed both RNGs with the group number ----
G = 178                             # our group number
random.seed(G)                      # seed python's random module
np.random.seed(G)                   # seed numpy's global RNG

# ---- Environment parameters derived from G ----
K = (G % 3) + 5                     # number of medicines (arms)
# Hidden success probability for each medicine i in {0,...,K-1}
true_probs = np.array([0.4 + ((G + i) % 6) * 0.07 for i in range(K)])

N_PATIENTS = 1000                   # total patients (iterations) to simulate

print('Group number G     :', G)
print('Total medicines K  :', K)
print('Hidden success probabilities (P_i):')
for i, p in enumerate(true_probs):
    print(f'  Medicine {i}: P = {p:.2f}')
print(f'\nTrue best medicine : {int(np.argmax(true_probs))} (P = {true_probs.max():.2f})')

In [ ]:
# ---- Build the static part of the dataset (patient_id + severity_score) ----
# assigned_medicine / clinical_outcome / utility_score are populated *dynamically*
# by each algorithm at run time, so the base table only holds the fixed fields.
patient_ids = np.arange(N_PATIENTS)             # 0 .. 999
severity_scores = (patient_ids % 5) + 1         # severity in range 1..5

dataset = pd.DataFrame({
    'patient_id': patient_ids,                  # patient index
    'severity_score': severity_scores,          # disease severity (1-5)
    'assigned_medicine': np.nan,                # filled during an algorithm run
    'clinical_outcome': np.nan,                 # filled during an algorithm run
    'utility_score': np.nan                     # filled during an algorithm run
})

print('First 10 dataset rows:')
print(dataset.head(10).to_string(index=False))

In [ ]:
def administer_treatment(medicine, severity):
    # Simulate giving `medicine` to a patient of the given `severity`.
    # Returns:
    #   clinical_outcome (int)  : 1 if patient recovers else 0  (Bernoulli with P_i)
    #   utility_score   (float) : reward = clinical_outcome * (1 - severity/10)
    # clinical_outcome is used to UPDATE the bandit estimates;
    # utility_score is used to ACCUMULATE the cumulative reward.
    clinical_outcome = 1 if np.random.random() < true_probs[medicine] else 0
    utility_score = clinical_outcome * (1 - severity / 10)
    return clinical_outcome, utility_score

## Task 2: Immediate Exploitation Strategy (1 Mark)

**Policy:** *"Once a treatment appears best, keep prescribing only that treatment."*

- **Warm-up:** test each medicine exactly **10 times** (`10 * K` patients).
- **Exploit:** afterwards always pick the medicine with the highest estimated recovery
  rate (`argmax Q`).

In [ ]:
def run_greedy(initial_pulls=10):
    # Greedy / immediate-exploitation strategy over N_PATIENTS patients.
    # Each medicine is tried `initial_pulls` times (warm-up); then the best-so-far
    # medicine is always selected.
    np.random.seed(G)                       # re-seed: every strategy faces same outcomes
    Q = np.zeros(K)                         # estimated recovery rate per medicine
    N = np.zeros(K)                         # number of times each medicine was used
    df = dataset.copy()                     # private copy of the dataset to populate
    cumulative = np.zeros(N_PATIENTS)       # cumulative reward after each patient
    best_hist = np.zeros(N_PATIENTS, int)   # running best-estimated arm after each patient
    total = 0.0

    for t in range(N_PATIENTS):
        severity = int(df.at[t, 'severity_score'])
        if t < initial_pulls * K:           # warm-up: round-robin over all medicines
            arm = t % K
        else:                               # exploit: pick best estimated medicine
            arm = int(np.argmax(Q))
        outcome, utility = administer_treatment(arm, severity)
        N[arm] += 1                         # update usage count
        Q[arm] += (outcome - Q[arm]) / N[arm]   # incremental sample-average update
        total += utility                    # accumulate utility reward
        df.at[t, 'assigned_medicine'] = arm
        df.at[t, 'clinical_outcome'] = outcome
        df.at[t, 'utility_score'] = utility
        cumulative[t] = total
        best_hist[t] = int(np.argmax(Q))    # which medicine currently looks best
    return {'name': 'Greedy', 'df': df, 'Q': Q, 'N': N,
            'cumulative': cumulative, 'best_hist': best_hist, 'total': total}

greedy_res = run_greedy(initial_pulls=10)
print('Greedy estimated recovery rates Q:', np.round(greedy_res['Q'], 3))
print('Greedy pulls per medicine        :', greedy_res['N'].astype(int))
print('Greedy chosen best medicine      :', int(np.argmax(greedy_res['Q'])))
print(f"Greedy cumulative reward (1000)  : {greedy_res['total']:.2f}")
print('\nFirst 10 populated rows:')
print(greedy_res['df'].head(10).to_string(index=False))

## Task 3: Controlled Clinical Trial - Epsilon-Greedy (1.5 Marks)

**Policy:** mostly give the current best treatment, but with probability `epsilon`
explore a random treatment to discover hidden opportunities.

Main run uses **epsilon = 0.10**; we also analyse **0.01** and **0.50**.

In [ ]:
def run_epsilon_greedy(epsilon):
    # Epsilon-greedy strategy: explore a random medicine with probability `epsilon`,
    # otherwise exploit the medicine with the highest estimated recovery rate.
    np.random.seed(G)                       # same outcome stream for a fair comparison
    Q = np.zeros(K)
    N = np.zeros(K)
    df = dataset.copy()
    cumulative = np.zeros(N_PATIENTS)
    best_hist = np.zeros(N_PATIENTS, int)
    total = 0.0

    for t in range(N_PATIENTS):
        severity = int(df.at[t, 'severity_score'])
        if np.random.random() < epsilon:    # explore: random medicine
            arm = np.random.randint(K)
        else:                               # exploit: best estimated medicine
            arm = int(np.argmax(Q))
        outcome, utility = administer_treatment(arm, severity)
        N[arm] += 1
        Q[arm] += (outcome - Q[arm]) / N[arm]
        total += utility
        df.at[t, 'assigned_medicine'] = arm
        df.at[t, 'clinical_outcome'] = outcome
        df.at[t, 'utility_score'] = utility
        cumulative[t] = total
        best_hist[t] = int(np.argmax(Q))
    return {'name': f'Eps-Greedy ({epsilon:.0%})', 'df': df, 'Q': Q, 'N': N,
            'cumulative': cumulative, 'best_hist': best_hist, 'total': total}

eps10_res = run_epsilon_greedy(0.10)        # main run: 10% exploration
print('Epsilon = 10%')
print('  Estimated Q       :', np.round(eps10_res['Q'], 3))
print('  Pulls per medicine:', eps10_res['N'].astype(int))
print(f"  Cumulative reward : {eps10_res['total']:.2f}")
print('  First 10 populated rows:')
print(eps10_res['df'].head(10).to_string(index=False))

In [ ]:
# ---- Sensitivity analysis: epsilon = 1%, 10%, 50% ----
eps01_res = run_epsilon_greedy(0.01)        # very little exploration
eps50_res = run_epsilon_greedy(0.50)        # heavy exploration

print('Effect of exploration rate on final cumulative reward:')
for res in (eps01_res, eps10_res, eps50_res):
    best = int(np.argmax(res['Q']))
    print(f"  {res['name']:>18s} -> reward = {res['total']:7.2f} | best medicine = {best}")

print('\nObservation:')
print('  * 1%  -> too little exploration; risks locking onto a sub-optimal medicine.')
print('  * 10% -> balanced; quickly finds the best medicine and mostly exploits it.')
print('  * 50% -> too much exploration; wastes ~half the patients on inferior medicines.')

## Task 4: Confidence-Based Strategy - UCB1 (1 Mark)

Select the arm maximising the UCB1 index:

$$ a_t = \arg\max_i \left( Q_i + \sqrt{\frac{2\ln t}{N_i}} \right) $$

Each medicine is tried once first so every `N_i > 0` before the bonus is used.

In [ ]:
def run_ucb1():
    # UCB1 confidence-based strategy. Medicines with fewer observations receive a
    # larger exploration bonus that shrinks as more evidence is collected.
    np.random.seed(G)
    Q = np.zeros(K)
    N = np.zeros(K)
    df = dataset.copy()
    cumulative = np.zeros(N_PATIENTS)
    best_hist = np.zeros(N_PATIENTS, int)
    total = 0.0

    for t in range(N_PATIENTS):
        severity = int(df.at[t, 'severity_score'])
        if t < K:                           # initialise: try each medicine once
            arm = t
        else:                               # pick the arm with the largest UCB1 index
            ucb_values = Q + np.sqrt(2 * np.log(t + 1) / N)
            arm = int(np.argmax(ucb_values))
        outcome, utility = administer_treatment(arm, severity)
        N[arm] += 1
        Q[arm] += (outcome - Q[arm]) / N[arm]
        total += utility
        df.at[t, 'assigned_medicine'] = arm
        df.at[t, 'clinical_outcome'] = outcome
        df.at[t, 'utility_score'] = utility
        cumulative[t] = total
        best_hist[t] = int(np.argmax(Q))
    return {'name': 'UCB1', 'df': df, 'Q': Q, 'N': N,
            'cumulative': cumulative, 'best_hist': best_hist, 'total': total}

ucb_res = run_ucb1()
print('UCB1 estimated recovery rates Q:', np.round(ucb_res['Q'], 3))
print('UCB1 pulls per medicine        :', ucb_res['N'].astype(int))
print('UCB1 chosen best medicine      :', int(np.argmax(ucb_res['Q'])))
print(f"UCB1 cumulative reward (1000)  : {ucb_res['total']:.2f}")
print('\nFirst 10 populated rows:')
print(ucb_res['df'].head(10).to_string(index=False))

## Task 5: Comparative Analysis (0.5 Mark)

Plot **Cumulative Reward vs. Number of Patients** for every strategy.

In [ ]:
# Compare every strategy on one cumulative-reward plot
strategies = [greedy_res, eps01_res, eps10_res, eps50_res, ucb_res]

plt.figure(figsize=(10, 6))
x = np.arange(1, N_PATIENTS + 1)            # patient index on the x-axis
for res in strategies:                      # one curve per strategy
    plt.plot(x, res['cumulative'], label=f"{res['name']} ({res['total']:.1f})")
plt.xlabel('Number of Patients')
plt.ylabel('Cumulative Reward (utility)')
plt.title('Cumulative Reward vs. Number of Patients (Group 178)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Final cumulative reward ranking:')
for res in sorted(strategies, key=lambda r: r['total'], reverse=True):
    print(f"  {res['name']:>18s} : {res['total']:7.2f}")

### Quantitative answers computed from the run

The cell below derives the answers to Questions 1-3 **directly from this run's data** so
the stated conclusions always match the numbers above.

- **Highest cumulative reward** = strategy with the largest final `total`.
- **Fastest convergence** = earliest patient index after which the running best-estimated
  medicine no longer changes (i.e. it has locked onto its final choice).
- **Most stable** = smallest standard deviation of the per-patient reward over the last
  200 patients (fewest fluctuations).

In [ ]:
true_best = int(np.argmax(true_probs))      # the genuinely optimal medicine (=1)

def convergence_point(res):
    # Earliest patient index after which the running best-estimated arm stays constant.
    bh = res['best_hist']
    final = bh[-1]
    conv = 0
    for t in range(N_PATIENTS - 1, -1, -1):
        if bh[t] == final:
            conv = t
        else:
            break
    return conv

def stability(res):
    # Std-dev of per-patient reward over the last 200 patients (lower = more stable).
    increments = np.diff(np.concatenate([[0.0], res['cumulative']]))
    return float(np.std(increments[-200:]))

print('Strategy            | final reward | best medicine | converged@ | stability(std)')
for res in strategies:
    print(f"  {res['name']:>16s} | {res['total']:11.2f} | "
          f"{int(np.argmax(res['Q'])):^13d} | {convergence_point(res):^10d} | "
          f"{stability(res):.4f}")

highest = max(strategies, key=lambda r: r['total'])
fastest = min(strategies, key=convergence_point)
most_stable = min(strategies, key=stability)
print('\nQ1 Highest cumulative reward :', highest['name'])
print('Q2 Fastest convergence       :', fastest['name'])
print('Q3 Most stable performance   :', most_stable['name'])
print('   True best medicine        :', true_best,
      '(P =', round(float(true_probs[true_best]), 2), ')')

### Analysis Questions & Comparative Summary

Questions 1-3 are answered by the computed cell above (so they always match this run).
The discussion below interprets those results.

**1. Highest cumulative reward at 1000 patients?**
See `Q1` above. The strategies that quickly lock onto the true best medicine
(medicine 1, P = 0.75) and then exploit it - typically UCB1 and Epsilon-Greedy (10%) -
accumulate the most reward, while Epsilon-Greedy (50%) loses reward to constant random
exploration.

**2. Fastest convergence?**
See `Q2` above. UCB1 usually converges earliest because its confidence bonus forces it to
disambiguate the arms early and then commit, without relying on random luck.

**3. Most stable performance?**
See `Q3` above. Greedy and UCB1 are the most stable once they commit (smooth, near-linear
curve); Epsilon-Greedy (50%) fluctuates the most.

**4. Safest approach for real-world hospital deployment?**
**UCB1.** It is deterministic (no random prescriptions), gives principled extra chances
to under-tested treatments, converges quickly, and offers confidence-based justification
for each choice - all valuable when patient safety matters.

**Comparative summary.**
Pure Greedy is efficient but risky: a poor warm-up can lock it onto a sub-optimal
medicine permanently (note how its convergence point and final medicine in the table can
differ from the true best). Epsilon-Greedy adds exploration; 10% performs well, 1% can get
stuck, and 50% wastes too many patients on inferior medicines. UCB1 gives the best balance
- fast, stable convergence to the true best medicine while still evaluating every
treatment fairly - making it the recommended choice for this clinical setting.